In [1]:
import numpy as np
import pandas as pd
import math
import os
import glob
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import clear_output, display
from scipy.signal import find_peaks  # 新增：用于特征提取

# 显式声明模式
%matplotlib inline 

# ==========================================
# 1. 基础路径配置与数据持久化加载
# ==========================================
base_path = r'D:\code\data\green'
search_pattern = os.path.join(base_path, '202*.csv')
file_list = sorted(glob.glob(search_pattern))

if not file_list:
    print("❌ 未找到数据文件，请检查路径！")
else:
    print("⏳ 正在读取并拼接原始 CSV 数据，请稍候...")
    df_list = [pd.read_csv(f) for f in file_list]
    df = pd.concat(df_list, ignore_index=True)
    
    # 清理多余列并处理缺失值
    columns_to_drop = ['clean_green1', 'butter_green1']
    df = df.drop(columns=[col for col in columns_to_drop if col in df.columns], errors='ignore')
    df['green1'] = df['green1'].ffill().fillna(0)  

    # 反转信号极性
    INVERT_POLARITY = True
    polarity_multiplier = -1.0 if INVERT_POLARITY else 1.0
    raw_signal = df['green1'].values * polarity_multiplier
    
    total_points = len(raw_signal)
    fs = 100  # 采样率 100Hz
    print(f"✅ 数据加载就绪！总行数: {total_points}，总时长: {total_points/fs:.1f} 秒")

⏳ 正在读取并拼接原始 CSV 数据，请稍候...
✅ 数据加载就绪！总行数: 8262961，总时长: 82629.6 秒


In [2]:
# ==========================================
# 2. 算法管线：滤波 + 寻峰过滤 + 基线提取
# ==========================================
from scipy.signal import butter, filtfilt

def run_one_pass_filter(raw_y, fs=100, window_sec=10.0, q=201, rho=0, overlap=0.5):
    chunk_size = int(fs * window_sec)
    step_size = int(chunk_size * (1 - overlap))
    window = np.hanning(chunk_size)
    t_chunk = np.linspace(0, window_sec, chunk_size, endpoint=False)
    Phi = np.zeros((chunk_size, q))
    Phi[:, 0] = 1 / np.sqrt(window_sec)
    W = np.zeros((q, q))
    for k in range(1, (q + 1) // 2):
        freq = k / window_sec
        Phi[:, 2 * k - 1] = np.sqrt(2 / window_sec) * np.cos(2 * np.pi * freq * t_chunk)
        Phi[:, 2 * k]     = np.sqrt(2 / window_sec) * np.sin(2 * np.pi * freq * t_chunk)
        penalty = (2 * np.pi * freq) ** 4
        W[2 * k - 1, 2 * k - 1] = penalty
        W[2 * k, 2 * k] = penalty
    H_hat = np.eye(q)
    inv_mat = np.linalg.inv(H_hat + rho * W)
    ac_signal = np.zeros_like(raw_y, dtype=np.float64)
    weight_sum = np.zeros_like(raw_y, dtype=np.float64)
    num_steps = max(1, math.ceil((len(raw_y) - chunk_size) / step_size) + 1)

    for i in range(num_steps):
        start_idx = i * step_size
        end_idx = min(start_idx + chunk_size, len(raw_y))
        current_len = end_idx - start_idx
        chunk_raw = raw_y[start_idx:end_idx]
        x_idx = np.arange(current_len)
        poly_coef = np.polyfit(x_idx, chunk_raw, 1) 
        trend = np.polyval(poly_coef, x_idx)        
        chunk_ac = chunk_raw - trend
        if current_len == chunk_size:
            G = Phi.T @ chunk_ac
            a_hat = inv_mat @ (G / chunk_size)
            reconstructed = Phi @ a_hat
            ac_signal[start_idx:end_idx] += reconstructed * window
            weight_sum[start_idx:end_idx] += window
        else:
            t_temp = t_chunk[:current_len]
            Phi_temp = np.zeros((current_len, q))
            Phi_temp[:, 0] = 1 / np.sqrt(window_sec)
            for k in range(1, (q + 1) // 2):
                freq = k / window_sec
                Phi_temp[:, 2 * k - 1] = np.sqrt(2 / window_sec) * np.cos(2 * np.pi * freq * t_temp)
                Phi_temp[:, 2 * k]     = np.sqrt(2 / window_sec) * np.sin(2 * np.pi * freq * t_temp)
            G_temp = Phi_temp.T @ chunk_ac
            a_hat_temp = inv_mat @ (G_temp / current_len)
            reconstructed = Phi_temp @ a_hat_temp
            window_temp = window[:current_len]
            ac_signal[start_idx:end_idx] += reconstructed * window_temp
            weight_sum[start_idx:end_idx] += window_temp

    mask = weight_sum > 1e-8
    ac_signal[mask] = ac_signal[mask] / weight_sum[mask]
    ac_signal[~mask] = 0.0 
    return ac_signal

def get_math_extrema(ac_signal):
    dy = np.diff(ac_signal)
    peaks = np.where((dy[:-1] > 0) & (dy[1:] <= 0))[0] + 1
    valleys = np.where((dy[:-1] < 0) & (dy[1:] >= 0))[0] + 1
    return peaks, valleys

def filter_and_score_extrema(ac_signal, raw_peaks, raw_valleys, fs=100):
    if len(raw_peaks) == 0 or len(raw_valleys) == 0:
        return np.array([]), np.array([]), np.array([])
        
    extrema = [(p, 'P') for p in raw_peaks] + [(v, 'V') for v in raw_valleys]
    extrema.sort(key=lambda x: x[0])

    def enforce_alternating(ext_list):
        if not ext_list: return []
        result = []
        current_type = ext_list[0][1]
        current_group = [ext_list[0]]
        for i in range(1, len(ext_list)):
            idx, p_type = ext_list[i]
            if p_type == current_type:
                current_group.append((idx, p_type))
            else:
                best_idx = max(current_group, key=lambda x: ac_signal[x[0]])[0] if current_type == 'P' else min(current_group, key=lambda x: ac_signal[x[0]])[0]
                result.append((best_idx, current_type))
                current_type = p_type
                current_group = [(idx, p_type)]
        if current_group:
            best_idx = max(current_group, key=lambda x: ac_signal[x[0]])[0] if current_type == 'P' else min(current_group, key=lambda x: ac_signal[x[0]])[0]
            result.append((best_idx, current_type))
        return result

    alt_extrema = enforce_alternating(extrema)
    temp_peaks = [x[0] for x in alt_extrema if x[1] == 'P']
    temp_valleys = [x[0] for x in alt_extrema if x[1] == 'V']

    p_amplitudes = {}
    for p in temp_peaks:
        prev_v = [v for v in temp_valleys if v < p]
        next_v = [v for v in temp_valleys if v > p]
        
        if prev_v and next_v:
            v1, v2 = prev_v[-1], next_v[0]
            slope = (ac_signal[v2] - ac_signal[v1]) / (v2 - v1)
            base_y = ac_signal[v1] + slope * (p - v1)
            amp = ac_signal[p] - base_y
        elif prev_v:
            amp = ac_signal[p] - ac_signal[prev_v[-1]]
        elif next_v:
            amp = ac_signal[p] - ac_signal[next_v[0]]
        else:
            amp = ac_signal[p]
            
        p_amplitudes[p] = max(1e-6, amp) 

    competition_window = int(fs * 0.30) 
    final_peaks = []
    
    for p in temp_peaks:
        if not final_peaks:
            final_peaks.append(p)
            continue
            
        last_p = final_peaks[-1]
        dist = p - last_p
        
        if dist < competition_window:
            if p_amplitudes[p] > p_amplitudes[last_p]:
                final_peaks[-1] = p 
        else:
            final_peaks.append(p)

    final_amps = [p_amplitudes[p] for p in final_peaks]
    if final_amps:
        median_amp = np.median(final_amps)
        if median_amp == 0: median_amp = 1e-6
    else:
        median_amp = 1e-6

    merged_extrema = [(p, 'P') for p in final_peaks] + [(v, 'V') for v in temp_valleys]
    merged_extrema.sort(key=lambda x: x[0])
    clean_extrema = enforce_alternating(merged_extrema) 

    out_peaks = [x[0] for x in clean_extrema if x[1] == 'P']
    out_valleys = [x[0] for x in clean_extrema if x[1] == 'V']
    
    out_scores = []
    for p in out_peaks:
        score = p_amplitudes[p] / (median_amp * 0.8)
        out_scores.append(min(1.0, max(0.01, score)))

    return np.array(out_peaks), np.array(out_valleys), np.array(out_scores)

def streaming_pipeline_with_filter(ac_signal, fs=100, chunk_sec=5.0):
    chunk_size = int(fs * chunk_sec)
    buffer_size = int(fs * 0.5)  
    num_steps = math.ceil(len(ac_signal) / chunk_size)

    all_peaks, all_valleys, all_scores = [], [], []
    print("⏳ 正在执行特征提取与智能过滤打分...")

    for i in range(num_steps):
        start_idx = i * chunk_size
        end_idx = min((i + 1) * chunk_size, len(ac_signal))
        buf_start = max(0, start_idx - buffer_size)
        buf_end = min(len(ac_signal), end_idx + buffer_size)
        
        if (buf_end - buf_start) < 3: continue
            
        current_chunk = ac_signal[buf_start:buf_end]
        raw_p_local, raw_v_local = get_math_extrema(current_chunk)
        raw_p_global = raw_p_local + buf_start
        raw_v_global = raw_v_local + buf_start
        
        filt_p, filt_v, scores = filter_and_score_extrema(ac_signal, raw_p_global, raw_v_global, fs)
        
        for p, score in zip(filt_p, scores):
            if start_idx <= p < end_idx:
                all_peaks.append(p)
                all_scores.append(score)
                
        for v in filt_v:
            if start_idx <= v < end_idx:
                all_valleys.append(v)

    print(f"✅ 智能过滤完成。保留了 {len(all_peaks)} 个有效心跳特征。")
    return np.array(all_peaks), np.array(all_valleys), np.array(all_scores)

# ✅ 新增：基线提取函数
def calculate_baseline(signal, fs=100, cutoff=0.5):
    """
    使用零相移低通滤波器提取基线。
    cutoff=0.5 Hz 能有效滤除心跳(>1Hz)，只保留缓慢的基线起伏。
    """
    nyq = 0.5 * fs
    b, a = butter(2, cutoff / nyq, btype='low')
    padlen = min(3 * max(len(b), len(a)), len(signal) - 1)
    # filtfilt 保证正反向滤波，零相位延迟，确保基线不会与原始波形产生时间错位
    return filtfilt(b, a, signal, padlen=padlen)

In [3]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output

# ==========================================
# 2.5 预处理：丢弃前 30s 的脏信号
# ==========================================
fs = 100  # 假设采样率为 100Hz
discard_sec = 30.0
discard_points = int(discard_sec * fs)

# 确保信号长度足够再进行切片
if 'raw_signal' in globals() and len(raw_signal) > discard_points:
    clean_raw_signal = raw_signal[discard_points:]
    time_offset = discard_sec
else:
    clean_raw_signal = raw_signal  # 如果信号不足30秒则不作处理
    time_offset = 0.0
    print("⚠️ 信号总长度不足 30 秒，跳过截断步骤。")

total_points = len(clean_raw_signal)

# ==========================================
# 3. 交互式调参器与特征可视化 (基线渲染版)
# ==========================================

_cached_q = None
_cached_rho = None
_cached_ac = None
_cached_peaks = None
_cached_valleys = None
_cached_scores = None

# ✅ 基线缓存
_cached_raw_baseline = None
_cached_ac_baseline = None

def interactive_tuner_plot(q, rho, start_idx, window_size):
    global _cached_q, _cached_rho, _cached_ac, _cached_peaks, _cached_valleys, _cached_scores
    global _cached_raw_baseline, _cached_ac_baseline
    
    clear_output(wait=True) 
    
    if _cached_ac is None or _cached_q != q or _cached_rho != rho:
        print(f"⚙️ 参数变更 (q={q}, rho={rho})，执行管线...")
        # ✅ 注意：这里传入的是剔除了前30s的 clean_raw_signal
        _cached_ac = run_one_pass_filter(clean_raw_signal, fs=100, window_sec=10.0, q=q, rho=rho)
        _cached_peaks, _cached_valleys, _cached_scores = streaming_pipeline_with_filter(_cached_ac, fs=100, chunk_sec=5.0)
        
        print("⏳ 正在计算原始信号与交流信号的 0.5Hz 基线...")
        _cached_raw_baseline = calculate_baseline(clean_raw_signal, fs=100)
        _cached_ac_baseline = calculate_baseline(_cached_ac, fs=100)
        
        _cached_q = q
        _cached_rho = rho
    
    end_idx = min(start_idx + window_size, total_points)
    if (end_idx - start_idx) < 10: return 

    # ✅ 时间轴加上 time_offset，保留原始录制的时间戳概念
    time_axis = (np.arange(start_idx, end_idx) / fs) + time_offset
    
    raw_slice = clean_raw_signal[start_idx:end_idx]
    ac_slice = _cached_ac[start_idx:end_idx]
    
    # 提取当前视图内的基线数据
    raw_base_slice = _cached_raw_baseline[start_idx:end_idx]
    ac_base_slice = _cached_ac_baseline[start_idx:end_idx]
    
    mask_p = (_cached_peaks >= start_idx) & (_cached_peaks < end_idx)
    view_peaks = _cached_peaks[mask_p]
    view_scores = _cached_scores[mask_p]
    
    mask_v = (_cached_valleys >= start_idx) & (_cached_valleys < end_idx)
    view_valleys = _cached_valleys[mask_v]
    
    plt.close('all') 
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(15, 8), sharex=True)
    
    # -------- 子图 1: 原始 PPG 与基线 --------
    ax1.plot(time_axis, raw_slice, color='gray', linewidth=1.5, label='Raw PPG Signal')
    ax1.plot(time_axis, raw_base_slice, color='darkorange', linewidth=2, linestyle='--', alpha=0.8, label='Raw Baseline')
    
    if len(view_valleys) > 0:
        # ✅ 注意这里 X 轴坐标需要加上 time_offset
        ax1.scatter((view_valleys/fs) + time_offset, clean_raw_signal[view_valleys], color='green', marker='.', s=50, zorder=5)
        
    ax1.set_title(f'PPG Feature Extractor (Time: {(start_idx/fs) + time_offset:.1f}s to {(end_idx/fs) + time_offset:.1f}s)', fontsize=14, pad=10)
    ax1.set_ylabel('Raw Amp', fontsize=12)
    ax1.legend(loc='upper right')
    ax1.grid(True, alpha=0.3)
    
    # -------- 子图 2: 滤波后的 AC 波形与基线 --------
    ax2.plot(time_axis, ac_slice, color='black', linewidth=1.5, alpha=0.7, label=f'AC Signal')
    ax2.plot(time_axis, ac_base_slice, color='darkorange', linewidth=2, linestyle='--', alpha=0.8, label='AC Baseline')
    
    if len(view_valleys) > 0:
        ax2.scatter((view_valleys/fs) + time_offset, _cached_ac[view_valleys], color='green', marker='v', s=60, label='Filtered Valleys', zorder=5)

    if len(view_peaks) > 0:
        for p, score in zip(view_peaks, view_scores):
            color = plt.cm.Reds(0.3 + score * 0.7) 
            size = 40 + score * 60
            p_time = (p/fs) + time_offset # ✅ 调整波峰 X 坐标
            
            ax2.scatter(p_time, _cached_ac[p], color=color, marker='^', s=size, edgecolors='black', linewidth=0.5, zorder=6)
            if score < 0.8: 
                ax2.text(p_time, _cached_ac[p] + 0.05*np.max(ac_slice), f"{score:.2f}", fontsize=8, color='blue', ha='center')

    ax2.scatter([], [], color='red', marker='^', s=80, edgecolors='black', label='Filtered Peaks (Confidence)')

    ax2.set_xlabel('Absolute Time (Seconds)', fontsize=12)
    ax2.set_ylabel('AC Amp', fontsize=12)
    ax2.axhline(0, color='gray', linestyle=':', linewidth=1.5, label='Absolute Zero') 
    
    handles, labels = ax2.get_legend_handles_labels()
    by_label = dict(zip(labels, handles))
    ax2.legend(by_label.values(), by_label.keys(), loc='upper right')
    ax2.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    plt.close(fig) 

# ================= 交互控件构建 =================
# ✅ 滑块的 max_start 基于截断后的 total_points 计算
max_start = max(0, total_points - 500) 

dropdown_q = widgets.Dropdown(options=[101, 201, 301, 401, 501], value=201, description='基底数量 q:')
dropdown_rho = widgets.Dropdown(
    options=[('0 (无惩罚)', 0), ('1e-6 (轻微)', 1e-6), ('2e-6', 2e-6), ('3e-6', 3e-6), ('4e-6', 4e-6), ('5e-6', 5e-6), ('1e-5', 1e-5)], 
    value=3e-6, description='惩罚系数 rho:'
)
slider_start = widgets.IntSlider(min=0, max=max_start, step=200, value=0, description='起点:', continuous_update=False, layout=widgets.Layout(width='400px'))
slider_window = widgets.IntSlider(min=200, max=5000, step=100, value=1000, description='窗口:', continuous_update=False, layout=widgets.Layout(width='400px'))

out = widgets.interactive_output(
    interactive_tuner_plot, 
    {'q': dropdown_q, 'rho': dropdown_rho, 'start_idx': slider_start, 'window_size': slider_window}
)

display(widgets.VBox([widgets.HBox([dropdown_q, dropdown_rho]), widgets.HBox([slider_start, slider_window])]), out)

Output()